### **Foundational Natural Language Processing(NLP)**

#### **1. Core Laboratory Objective & Overview**
Before engineering large language models, practitioners must master how unstructured text is normalized, annotated, and parsed into predictable memory blocks. This practical introduces students to industry-grade text pipelines. Rather than treating text as a simple character string, students will programmatically extract dense linguistic annotations and construct deterministic text chunking mechanisms. This sequence serves as the essential ingestion data layer required later for building Retrieval-Augmented Generation (RAG) frameworks.

#### **2. Dataset / Test Data Constraints**
* **Primary Corpus:** Raw, multi-paragraph text samples extracted directly from standard NLTK corpora (e.g., `nltk.corpus.reuters` or `nltk.corpus.gutenberg`).
* **Structural Constraints:** The target source can also be a custom, unformatted 500-word uncompressed text file containing complex formatting, multi-sentence paragraphs, diverse numerical datatypes, and highly explicit named entity clusters (e.g., specific dates, geographic locations, and corporate organization acronyms).

#### **3. Core Implementation Tasks (Step-by-Step)**
Students must compose and execute an object-oriented or procedural python script completing the following operational phases:
1. **Pipeline Initialization:** Programmatically import and initialize an optimized, multi-task processing engine container in spaCy (such as the optimized `en_core_web_sm` or medium statistical `en_core_web_md` model weights).
2. **Linguistic Annotation Extraction:** Process the selected raw text corpus through the pipeline framework. Parse the output token elements to programmatically capture individual token text attributes, structural Part-of-Speech (`token.pos_`) tags, and exact syntactic dependency labels (`token.dep_`).
3. **Named Entity Recognition (NER) Isolation:** Traverse the instantiated document object's native entity array (`doc.ents`). Extract isolated sub-string entity tokens, map their global character slice indices, and resolve their corresponding semantic classifications (e.g., tracking `ORG`, `PERSON`, `GPE`, or `DATE`).
4. **Structural Text Chunking Loop:** Write a native python algorithm to partition the document stream into explicit, standardized physical blocks. Implement two optional variations:
   * *Sentence-Boundary Split:* Token slicing segmented strictly by linguistic terminal punctuation properties.
   * *Fixed-Size Windowing:* Generating fixed-length token groups bound by a predefined ceiling metric (e.g., 50 tokens) containing an explicit rolling overlapping boundary (e.g., a 10-token overlap step).
5. **Entity Visualization Layer:** Integrate and invoke spaCy's native `displacy` visual utility wrapper inside a local Jupyter Notebook session to generate real-time, inline web-rendered visualizations highlighting the extracted entity token spans.



In [ ]:
import spacy
from spacy import displacy
import pandas as pd

# ------------------------------------------------------------------
# 1. Pipeline Initialization
# ------------------------------------------------------------------
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download

    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

text_corpus = (
    "In Jan 2024, Dr. Smith joined OpenAI in San Francisco, U.S.A. "
    "He invested $50,000 in cloud architecture to build RAG pipelines. "
    "Natural Language Processing relies heavily on precise tokenization, "
    "Part-of-Speech tagging, and Named Entity Recognition. "
    "According to reports from Apple Inc., robust structural chunking prevents "
    "error propagation across downstream transformer models."
)

doc = nlp(text_corpus)

# ------------------------------------------------------------------
# 2. Linguistic Annotation Extraction
# ------------------------------------------------------------------
token_data = []
for token in doc:
    token_data.append(
        {
            "Text": token.text,
            "Lemma": token.lemma_,
            "POS": token.pos_,
            "Tag": token.tag_,
            "Dependency": token.dep_,
            "Head": token.head.text,
            "Is Stop": token.is_stop,
        }
    )

df_tokens = pd.DataFrame(token_data)
print("=== LINGUISTIC ANNOTATIONS (FIRST 10 TOKENS) ===")
print(df_tokens.head(10).to_string(index=False))
print("-" * 60)

# ------------------------------------------------------------------
# 3. Named Entity Recognition (NER) Isolation
# ------------------------------------------------------------------
entity_data = []
for ent in doc.ents:
    entity_data.append(
        {
            "Entity Text": ent.text,
            "Label": ent.label_,
            "Start Char": ent.start_char,
            "End Char": ent.end_char,
        }
    )

df_entities = pd.DataFrame(entity_data)
print("\n=== NAMED ENTITIES DETECTED ===")
print(df_entities.to_string(index=False))
print("-" * 60)


# ------------------------------------------------------------------
# 4. Structural Text Chunking
# ------------------------------------------------------------------
def chunk_by_sentences(spacy_doc):
    # Slices text strictly by sentence boundaries detected by spaCy parser
    return [sent.text.strip() for sent in spacy_doc.sents]


def chunk_by_fixed_window(spacy_doc, window_size=15, overlap=5):
    # Generates overlapping fixed-token sliding windows
    tokens = [token.text for token in spacy_doc]
    chunks = []
    step = window_size - overlap

    for i in range(0, len(tokens), step):
        chunk = " ".join(tokens[i : i + window_size])
        chunks.append(chunk)
        if i + window_size >= len(tokens):
            break

    return chunks


sentence_chunks = chunk_by_sentences(doc)
window_chunks = chunk_by_fixed_window(doc, window_size=15, overlap=5)

print("\n=== SENTENCE-BOUNDARY CHUNKS ===")
for idx, chunk in enumerate(sentence_chunks, 1):
    print(f"Chunk {idx}: {chunk}")

print("\n=== FIXED-SIZE SLIDING WINDOW CHUNKS (Size=15, Overlap=5) ===")
for idx, chunk in enumerate(window_chunks, 1):
    print(f"Chunk {idx}: {chunk}")

# ------------------------------------------------------------------
# 5. Entity Visualization Layer
# ------------------------------------------------------------------
displacy.render(doc, style="ent", jupyter=True)

=== LINGUISTIC ANNOTATIONS (FIRST 10 TOKENS) ===
  Text  Lemma   POS Tag Dependency      Head  Is Stop
    In     in   ADP  IN       prep    joined     True
   Jan    Jan PROPN NNP       pobj        In    False
  2024   2024   NUM  CD     nummod       Jan    False
     ,      , PUNCT   ,      punct    joined    False
   Dr.    Dr. PROPN NNP   compound     Smith    False
 Smith  Smith PROPN NNP      nsubj    joined    False
joined   join  VERB VBD       ROOT    joined    False
OpenAI OpenAI PROPN NNP       dobj    joined    False
    in     in   ADP  IN       prep    joined     True
   San    San PROPN NNP   compound Francisco    False
------------------------------------------------------------

=== NAMED ENTITIES DETECTED ===
       Entity Text  Label  Start Char  End Char
          Jan 2024   DATE           3        11
             Smith PERSON          17        22
            OpenAI    ORG          30        36
     San Francisco    GPE          40        53
            U.S.A.    G

#### **4. Post-Laboratory Work & Submission Deliverables**
Students must generate and compile a formal lab package containing:
* A fully functional, cell-executed Jupyter Notebook (`.ipynb`) containing explicit print-log verifications for each of the core implementation phases.
* **Internal Data Structure Explanation:** Code comments or Markdown descriptions detailing exactly how spaCy's primary data abstraction layer (the `Doc` container object) efficiently handles reference structures for individual underlying token arrays, spans, and metadata arrays.
* **Chunking Matrix Display:** Printed tabular results displaying the segmented text chunks alongside a side-by-side performance grid demonstrating how edge-case punctuation rules (e.g., abbreviations like *"U.S.A."* or *"Dr."*) alter the behavior of sentence boundaries.
* **Analytical Reflection Summary:** A brief technical assessment explaining the cascading mechanics of *Error Propagation* in industrial text workflows. Students must explicitly answer: *How does an error at an early phase of execution (e.g., an incorrect Part-of-Speech prediction) trigger systemic degradation in subsequent downstream layers like Named Entity Recognition or Causal Text Generation?*

#### **5. Advanced / Practice Exercises (Optional for Practice)**
* **Cross-Library Architecture Benchmark:** Pass the identical unformatted text payload through an un-pipelined, manual loop structure inside NLTK. Calculate and plot the exact performance variance regarding system processing time, memory allocation, and entity matching recall benchmarks compared directly against spaCy's integrated transformer-ready architecture.
* **Deterministic Privacy-Preserving Masking:** Develop a programmatic security script utilizing the mapped NER character index slices to automatically sanitize private text payloads. Write a routine that dynamically targets all detected instances of `PERSON`, `ORG`, or highly sensitive entity metadata arrays, overwriting those unique tokens with a standardized, anonymous string tag such as `[REDACTED]`.

In [ ]:
import time
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download(
    "averaged_perceptron_tagger_eng", quiet=True
)
nltk.download("maxent_ne_chunker", quiet=True)
nltk.download("maxent_ne_chunker_tab", quiet=True)
nltk.download("words", quiet=True)

# ------------------------------------------------------------------
# 1. spaCy vs NLTK
# ------------------------------------------------------------------
large_payload = text_corpus * 100

# Benchmark spaCy
start_spacy = time.perf_counter()
spacy_doc = nlp(large_payload)
spacy_entities = [(ent.text, ent.label_) for ent in spacy_doc.ents]
spacy_time = time.perf_counter() - start_spacy

# Benchmark NLTK
start_nltk = time.perf_counter()
nltk_tokens = word_tokenize(large_payload)
nltk_pos = pos_tag(nltk_tokens)
nltk_tree = ne_chunk(nltk_pos)

nltk_entities = []
for chunk in nltk_tree:
    if hasattr(chunk, "label"):
        nltk_entities.append((" ".join(c[0] for c in chunk), chunk.label()))
nltk_time = time.perf_counter() - start_nltk

print("=== PERFORMANCE BENCHMARK (100x Payload) ===")
print(
    f"spaCy Processing Time : {spacy_time:.4f} seconds | Entities Found: {len(spacy_entities)}"
)
print(
    f"NLTK Processing Time  : {nltk_time:.4f} seconds | Entities Found: {len(nltk_entities)}"
)
print("-" * 60)


# ------------------------------------------------------------------
# 2. Privacy-Preserving Masking Function
# ------------------------------------------------------------------
def sanitize_sensitive_entities(
    text, target_labels={"PERSON", "ORG", "GPE", "DATE", "MONEY"}
):
    doc_obj = nlp(text)
    sanitized_text = text

    entities_sorted = sorted(doc_obj.ents, key=lambda ent: ent.start_char, reverse=True)

    for ent in entities_sorted:
        if ent.label_ in target_labels:
            mask_tag = f"[{ent.label_}_REDACTED]"
            sanitized_text = (
                sanitized_text[: ent.start_char]
                + mask_tag
                + sanitized_text[ent.end_char :]
            )

    return sanitized_text


masked_output = sanitize_sensitive_entities(text_corpus)

print("\n=== PRIVACY-PRESERVING MASKING OUTPUT ===")
print("Original Text:")
print(text_corpus)
print("\nSanitized Text:")
print(masked_output)

=== PERFORMANCE BENCHMARK (100x Payload) ===
spaCy Processing Time : 0.7600 seconds | Entities Found: 900
NLTK Processing Time  : 0.8585 seconds | Entities Found: 700
------------------------------------------------------------

=== PRIVACY-PRESERVING MASKING OUTPUT ===
Original Text:
In Jan 2024, Dr. Smith joined OpenAI in San Francisco, U.S.A. He invested $50,000 in cloud architecture to build RAG pipelines. Natural Language Processing relies heavily on precise tokenization, Part-of-Speech tagging, and Named Entity Recognition. According to reports from Apple Inc., robust structural chunking prevents error propagation across downstream transformer models.

Sanitized Text:
In [DATE_REDACTED], Dr. [PERSON_REDACTED] joined [ORG_REDACTED] in [GPE_REDACTED], [GPE_REDACTED] He invested $[MONEY_REDACTED] in cloud architecture to build [ORG_REDACTED] pipelines. Natural Language Processing relies heavily on precise tokenization, Part-of-Speech tagging, and Named [PERSON_REDACTED]. According t